# Aggregate results — across-category comparison

Reads every `results/<model>_<wrapper>_<ticker>_<regime>.csv`, computes
(i) overall, (ii) in-crisis, (iii) out-of-crisis metrics, writes
`results/metrics_long.csv`, and renders the cross-method scatter into
`report/figures/aggregate/`.

In [ ]:
# sys.path bootstrap so 'from src...' works no matter where the kernel was launched from
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


In [ ]:
import warnings, glob, subprocess, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from src.analysis._shared import RESULTS_DIR, CRISIS_WINDOWS, MODEL_CATEGORIES
from src.plotting import plot_interval_timeline, compute_cell_metrics, render_cell_grid


## 1. Load every interval CSV

In [ ]:
# Aggregate across every CSV in results/
from src.analysis._shared import iterate_experiments

all_files = sorted(RESULTS_DIR.glob("*.csv"))
all_files = [f for f in all_files if f.name != "metrics_long.csv"]
print(f"Loaded {len(all_files)} interval CSVs")
raw = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in all_files], ignore_index=True)
raw["category"] = raw["model"].map(MODEL_CATEGORIES)
print(raw.groupby(["category", "model", "wrapper"]).size().to_string())


## 2. Compute per-cell metrics for overall / in-crisis / out-of-crisis

In [ ]:
# Per-cell metrics: overall, in-crisis, out-of-crisis
def metrics_for_subset(df, alpha=0.10):
    if len(df) == 0: return None
    realized = df['r'].astype(float).values
    lo = df['l'].astype(float).values
    up = df['u'].astype(float).values
    width_t = up - lo
    pen_lo = (2.0/alpha) * np.maximum(lo - realized, 0.0)
    pen_hi = (2.0/alpha) * np.maximum(realized - up, 0.0)
    in_iv = (realized >= lo) & (realized <= up)
    return {"cov": float(in_iv.mean()), "winkler": float(np.mean(width_t + pen_lo + pen_hi)),
            "width": float(np.mean(width_t)), "n": int(len(df))}

rows = []
for (model, wrapper, ticker, regime), grp in raw.groupby(["model", "wrapper", "ticker", "regime"]):
    cfg = CRISIS_WINDOWS[str(regime)]
    in_crisis  = grp[(grp["date"] >= cfg["onset"]) & (grp["date"] <= cfg["end"])]
    out_crisis = grp[(grp["date"] < cfg["onset"]) | (grp["date"] > cfg["end"])]
    overall = metrics_for_subset(grp)
    incr    = metrics_for_subset(in_crisis)
    outcr   = metrics_for_subset(out_crisis)
    if overall is None: continue
    rows.append({
        "category": MODEL_CATEGORIES.get(model, "unknown"),
        "model": model, "wrapper": wrapper, "ticker": ticker, "regime": regime,
        "cov_all": overall["cov"], "winkler_all": overall["winkler"], "width_all": overall["width"],
        "cov_in":  incr["cov"]    if incr  else np.nan, "winkler_in":  incr["winkler"]  if incr  else np.nan, "width_in":  incr["width"]  if incr  else np.nan,
        "cov_out": outcr["cov"]   if outcr else np.nan, "winkler_out": outcr["winkler"] if outcr else np.nan, "width_out": outcr["width"] if outcr else np.nan,
    })
metrics_long = pd.DataFrame(rows)
metrics_long.to_csv(RESULTS_DIR / "metrics_long.csv", index=False)
print(f"wrote {RESULTS_DIR/'metrics_long.csv'} ({len(metrics_long)} rows)")


## 3. Summary tables

In [ ]:
# Three summary tables: overall, in-crisis, out-of-crisis (sorted by Winkler)
def make_table(df, suffix):
    cols = [f"cov_{suffix}", f"winkler_{suffix}", f"width_{suffix}"]
    out = (df.groupby(["category", "model", "wrapper"])[cols]
             .mean()
             .sort_values(f"winkler_{suffix}")
             .round(3))
    return out

print("=" * 70)
print("Table 1 — Overall (all test-window days)")
print("=" * 70)
display(make_table(metrics_long, "all"))

print("\n" + "=" * 70)
print("Table 2 — In-crisis only")
print("=" * 70)
display(make_table(metrics_long, "in"))

print("\n" + "=" * 70)
print("Table 3 — Out-of-crisis (pre / post-crisis padding)")
print("=" * 70)
display(make_table(metrics_long, "out"))


## 4. Cross-method scatter — coverage gap vs Winkler

In [ ]:
# Cross-method scatter: |coverage gap| vs Winkler, coloured by category
fig, ax = plt.subplots(figsize=(9, 6))
agg = (metrics_long.groupby(["category", "model", "wrapper"])
       .agg(cov_all=("cov_all", "mean"), winkler=("winkler_all", "mean"))
       .reset_index())
agg["cov_gap"] = agg["cov_all"] - 0.90

CAT_COLOR = {"baseline":"#444444", "simple_native":"#9467bd", "simple_cp":"#2ca02c", "complex_cp":"#ff7f0e"}
for cat, sub in agg.groupby("category"):
    ax.scatter(sub["cov_gap"], sub["winkler"], s=110, alpha=0.85,
               edgecolor="black", linewidth=0.5,
               color=CAT_COLOR.get(cat, "C0"), label=cat)
    for _, row in sub.iterrows():
        ax.annotate(f"{row['model']}×{row['wrapper']}", (row["cov_gap"], row["winkler"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=7, alpha=0.75)
ax.axvline(0, color="black", linestyle=":", alpha=0.5)
ax.set_xlabel("Coverage gap from nominal 0.90  (negative = undercoverage)")
ax.set_ylabel("Winkler score (lower = better)")
ax.set_title("Method comparison — coverage calibration vs interval quality")
ax.grid(True, alpha=0.25)
ax.legend(title="Category", loc="upper right")
out = ROOT / "report" / "figures" / "aggregate" / "scatter_cov_winkler.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=140, bbox_inches="tight")
fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
print(f"saved → {out.relative_to(ROOT)}")
plt.show()
